In [4]:
import torch
import torch.nn.functional as F

kelimeler = ["Banka", "nehrin", "kenarındaydı"]

torch.manual_seed(42)
embeddings = torch.randn(3, 8)

print("Kelime vektörleri (embedding):")
for kelime, vektor in zip(kelimeler, embeddings):
    print(f"{kelime}: {vektor}")

Kelime vektörleri (embedding):
Banka: tensor([ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047])
nehrin: tensor([ 0.3559, -0.6866, -0.4934,  0.2415, -1.1109,  0.0915, -2.3169, -0.2168])
kenarındaydı: tensor([-0.3097, -0.3957,  0.8034, -0.6216, -0.5920, -0.0631, -0.8286,  0.3309])


In [5]:
Q = embeddings
K = embeddings
V = embeddings

skorlar = torch.matmul(Q, K.T)
print("Ham dikkat skorları (kelimelerin birbirine ne kadar 'benzediği'):")
print(skorlar)

skorlar_olceklendi = skorlar / (8 ** 0.5)

dikkat_agirliklari = F.softmax(skorlar_olceklendi, dim=-1)
print("\nDikkat ağırlıkları (her satır toplamı 1.0 olur):")
print(dikkat_agirliklari)

sonuc = torch.matmul(dikkat_agirliklari, V)
print("\nBağlama duyarlı yeni vektörler:")
print(sonuc)

Ham dikkat skorları (kelimelerin birbirine ne kadar 'benzediği'):
tensor([[15.7307, -1.7073,  0.0280],
        [-1.7073,  7.5574,  2.1148],
        [ 0.0280,  2.1148,  2.4348]])

Dikkat ağırlıkları (her satır toplamı 1.0 olur):
tensor([[0.9941, 0.0021, 0.0039],
        [0.0319, 0.8448, 0.1233],
        [0.1841, 0.3849, 0.4310]])

Bağlama duyarlı yeni vektörler:
tensor([[ 1.9150,  1.4755,  0.8974, -2.0949,  0.6698, -1.2273, -0.0508, -1.5943],
        [ 0.3239, -0.5813, -0.2889,  0.0601, -0.9898,  0.0301, -2.0608, -0.1936],
        [ 0.3581, -0.1611,  0.3222, -0.5625, -0.5579, -0.2192, -1.2569, -0.2362]])


## 🧩 Transformer Mimarisine Giriş (LLM'lerin Temeli)

**Amaç:** Bugünkü LLM'lerin (ChatGPT, Claude gibi) temelinde yatan Transformer mimarisini, kavramsal olarak ve küçük bir kod örneğiyle öğrenmek.

**Öğrenilen kavramlar:**

1. **RNN'in sınırlılığı:** Eski yöntemler (RNN) cümleyi kelime kelime, sırayla okuyordu — cümle uzadıkça başındaki kelimeleri "unutma" problemi yaşıyordu.

2. **Tokenization:** Metni küçük parçalara (token) bölüp, her parçaya bir sayı (ID) atama işlemi.

3. **Embedding:** Her token ID'sini, anlamını taşıyan bir sayı vektörüne çevirme — anlamca yakın kelimeler, vektör uzayında da birbirine yakın konumlanıyor (örn. Kral - Erkek + Kadın ≈ Kraliçe).

4. **Self-Attention:** Her kelimenin, cümledeki diğer tüm kelimelere aynı anda bakıp, bağlamı netleştirmek için "dikkat puanı" hesaplaması (örn. "Banka nehrin kenarındaydı" cümlesinde "banka" kelimesinin "nehrin" kelimesine yüksek dikkat vermesi).

5. **Multi-Head Attention:** Aynı attention işleminin, farklı ilişki türlerine (özne, nesne, bağlam vb.) odaklanan birden fazla "kafa" ile paralel çalıştırılması.

6. **Positional Encoding:** Attention kelimelerin sırasını bilmediği için, her kelimenin vektörüne konum bilgisinin ayrıca eklenmesi.

**Kodla Denenen Örnek:**
PyTorch ile küçük bir cümle (3 kelime) üzerinde, elle self-attention hesaplaması yapıldı:
- Rastgele embedding vektörleri oluşturuldu
- Query-Key çarpımıyla ham dikkat skorları hesaplandı
- Skorlar ölçeklendirilip Softmax ile dikkat ağırlıklarına (toplamı 1 olan yüzdelere) çevrildi
- Bu ağırlıklarla, her kelimenin bağlama duyarlı yeni vektörü elde edildi

**Genel akış:**
Tokenization → Embedding → (+Positional Encoding) → Multi-Head Self-Attention → Feed-Forward katmanlar → Çıktı

**Çıkardığım ders:** Attention mekanizması, CNN'deki convolution'a benzer bir mantıkla çalışıyor — CNN bir görüntüdeki komşu pikselleri tararken, attention bir cümledeki tüm kelimeler arasındaki ilişkiyi aynı anda tarıyor. İkisi de "hangi bilginin önemli olduğunu otomatik keşfetme" fikrine dayanıyor.